## ADP

**ADP** (Aerosol Data Protocol) is a protocol developed by Aerosol d.o.o. for standardized data exchange between instruments and data acquisition systems. It allows for real-time data retrieval, control and monitoring of compatible instruments.

ADP works with **AE36s**, **AE36**, **TCA08** and **TCA09** instruments, while **AE33 is not supported** (AE33 uses legacy AE33 data protocol, which behaves similarly, only commands are different).

**License:** Aerosol Magee Scientific Software License (See LICENSE file for full terms).

In [10]:
from datetime import timedelta, datetime
from io import StringIO

import pandas as pd

from aerosol_magee_pytools.data_access.tcp_ip import request_tcp, guess_delimiter
from aerosol_magee_pytools.instruments.column_names import COLUMNS_AE36S, COLUMNS_TCA09, COLUMNS_TCA08
from aerosol_magee_pytools.tools import dotnet_seconds_to_datetime, datetime_to_dotnet_nanoseconds

In [11]:
# IP of the instrument - change it to the actual IP address of your AE36s instrument
instrument_ip = '10.10.10.80' # Skylab AE36s
# instrument_ip = '10.10.10.133' # Skylab TCA08
# instrument_ip = '10.10.10.61' # Skylab TCA09

timeout = 2.0  # seconds

In [12]:
### command INFO
command = '$AERO:INFO\r\n'
received_text = request_tcp(ip=instrument_ip,
                            command=command,
                            timeout=timeout)
print(received_text)

# parse info response to a dictionary
info = {}
for line in received_text.splitlines():
    line = line.strip()
    if not line or ':' not in line:
        continue
    key, value = line.split(':', 1)
    info[key.strip()] = value.strip()

# there are different fields in INFO response between TCA and AE36, let's synchronize info
if 'Serialnumber' in info:
    info['Serial Number'] = info.pop('Serialnumber' )
    info['Model number'] = info['Serial Number'].split('-')[0]

instrument_typ = info['Model number']
print()
print(f"Instrument type: {instrument_typ}")
print()
print(info)

Connection information
Serialnumber: AE36s-00-00107
Connection ID: 1010
Connection time: 23-Jun-2026 11:03:29
CPU used: 9 % , Memory used: 24 %

Instrument type: AE36s

{'Connection ID': '1010', 'Connection time': '23-Jun-2026 11:03:29', 'CPU used': '9 % , Memory used: 24 %', 'Serial Number': 'AE36s-00-00107', 'Model number': 'AE36s'}


In [13]:
### command LAST return entry from chosen table (table DATA in our case)
command = '$AERO:LAST DATA\r\n'
received_text = request_tcp(ip=instrument_ip,
                            command=command,
                            timeout=timeout)
print(received_text)

5012058,639178021800000000,639178093800000000,56,1,1,0,0,0,0,1,0,10,10,0,S,871476.625,310321.875,609796.625,835692.375,295306.75,575369.625,839922.875,325270.375,593993,850120.625,341860.375,581751.75,853644,369300.25,588934.125,876736,457115.25,693710.375,864309.625,431501.75,614264.625,862050.25,510721.5,689959.375,857397.75,484355,638719.125,76.17249,24.89891,81.8105,26.51046,75.32765,24.0425,65.79124,20.45982,58.02211,17.6787,50.94353,15.1609,47.77012,14.15014,34.74821,9.766956,32.41128,9.090624,615,741,833,680,800,912,675,780,875,726,863,879,738,773,847,751,759,812,771,810,820,844,827,785,883,889,815,0.003441493,0.003112793,0.003035523,0.002647814,0.002206295,0.001452551,0.001254084,-0.002157055,-0.002580738,7.8,16.75255,16.85094,14.95848,12.78534,11.12471,9.397504,8.894008,6.098912,5.861235,0.9671316,2.34542,1.540874,1.366096,0.8034754,0.3008213,0.3748922,0,0,342,101325,25,3766,1228,4994,2506,162,30.3,52.4,29.3,28.4,31.4,34.3,33.4,1388,279,9


**FETCH** command allows you to retrieve data from the instrument for a specific time period.

**Note:** On AE36, AE36s and TCA09, the data is stored in the table named `Data`, while on TCA08, data is stored in the table named `OnlineResults` (table named `Data` on TCA08 is acctually debug_data).

In [14]:
### command FETCH return data from chosen table and time period of last 10 minutes

#### limit time to resent data
end = datetime.now().isoformat(sep=' ', timespec='seconds')
start = (datetime.now() - timedelta(hours=6)).isoformat(sep=' ', timespec='seconds')
###

if instrument_typ != 'TCA08':
    # command = '$AERO:FETCH DATA "2026-06-11 12:51:00" "2026-06-11 15:51:00"\r\n'
    command = f'$AERO:FETCH DATA "{start}" "{end}"\r\n'
else:
    # command = '$AERO:FETCH OnlineResults "2026-06-11 12:51:00" "2026-06-11 15:51:00
    command = f'$AERO:FETCH OnlineResult "{start}" "{end}"\r\n'

received_text = request_tcp(ip=instrument_ip,
                            command=command,
                            timeout=timeout)
print(received_text)

5011699,639177806400000000,639177878400000000,56,1,1,0,0,0,0,1,0,10,10,0,0,879554.75,472831,710050.25,839280.75,461190.125,673169.375,842834.5,495370.25,685832.375,853635.625,498534.25,659580.625,856536.25,516731.875,656394.75,895779.5,627288.5,776021.625,867365.625,571643.125,670773.25,871834,634975.125,740009,868043.875,594937.5,683127,34.98236,10.60056,37.6594,11.24049,33.60883,10.01199,28.47686,8.316397,24.76893,7.172138,21.44527,6.097151,19.99874,5.702557,14.10062,3.892542,13.08149,3.60306,771,770,870,909,901,1026,918,902,1025,930,872,1011,913,836,973,914,815,952,917,808,949,923,824,909,910,747,891,0.003239915,0.00302321,0.003104561,0.002793646,0.002478351,0.00185366,0.00170987,-0.001129913,-0.001586902,13.1,17.48717,18.9502,17.5149,14.69279,12.7844,11.02389,10.28737,7.062416,6.409062,-0.7920265,2.153101,1.977581,1.469546,0.8326159,0.4901133,0.4224062,0,0,439,101325,25,3786,1209,4995,2490,160,28.6,56.5,27.1,29.3,29.9,32.8,32,1388,277,8
5011700,639177807000000000,639177879000000000

In [15]:
# in the end, you can parse data, for example, convert it into pandas dataframe;
# you will need column names first

if instrument_typ in ['AE36', 'AE36s']:
    columns = COLUMNS_AE36S['Data']
elif instrument_typ == 'TCA08':
    columns = COLUMNS_TCA08['OnlineResult']
elif instrument_typ == 'TCA09':
    columns = COLUMNS_TCA09['Data']
else:
    raise ValueError(f"Unsupported instrument type: {instrument_typ}")

# python can automatically determine the delimiter
separator = guess_delimiter(received_text)

df_data = pd.read_csv(StringIO(received_text.strip()),
                      sep=separator,
                      names=columns)

# convert instrument timestamps stored as .NET epoch seconds to pandas datetime
for column in df_data.columns:
    if ('Timestamp' in column) or column.endswith('TimeUTC') or column.endswith('TimeLocal'):
        if instrument_typ != 'TCA08':
            df_data[column] = dotnet_seconds_to_datetime(df_data[column])
        else:
            df_data[column] = pd.to_datetime(df_data[column])

print(df_data.shape)
print(df_data)

(360, 134)
          ID        TimestampUTC      TimestampLocal  SetupID  G0_Status  \
0    5011699 2026-06-23 03:04:00 2026-06-23 05:04:00       56          1   
1    5011700 2026-06-23 03:05:00 2026-06-23 05:05:00       56          1   
2    5011701 2026-06-23 03:06:00 2026-06-23 05:06:00       56          1   
3    5011702 2026-06-23 03:07:00 2026-06-23 05:07:00       56          1   
4    5011703 2026-06-23 03:08:00 2026-06-23 05:08:00       56          1   
..       ...                 ...                 ...      ...        ...   
355  5012054 2026-06-23 08:59:00 2026-06-23 10:59:00       56          1   
356  5012055 2026-06-23 09:00:00 2026-06-23 11:00:00       56          1   
357  5012056 2026-06-23 09:01:00 2026-06-23 11:01:00       56          1   
358  5012057 2026-06-23 09:02:00 2026-06-23 11:02:00       56          1   
359  5012058 2026-06-23 09:03:00 2026-06-23 11:03:00       56          1   

     G1_Status  G2_Status  G3_Status  G4_Status  G5_Status  G6_Status  \
0  